# 02 · Dataset loading for the model (B4)

Validation that the processed PHQ-8 item dataset is ready for modeling.

**Done when:**
1. `load_item_dataset` works.
2. Required columns exist: `item_text`, `transcript_text`, `label`, `split`.
3. A single row can be pulled and inspected.
4. `item_text` + evidence can be passed to a BERT tokenizer as a sentence pair.

We use `phq8_item_dataset_full.csv`, which adds `split` and the evidence columns
(`transcript_text`, `baseline_utterances`, `retrieved_utterances`) on top of the base item dataset.


## 0 · Setup


In [ ]:
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "src").exists() else CWD.parent
assert (PROJECT_ROOT / "src").exists(), f"Could not locate project root from {CWD}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset_loader import load_item_dataset
from src.models.input_formatting import format_model_input, split_utterances

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "phq8_item_dataset_full.csv"
print("Project root :", PROJECT_ROOT)
print("Dataset path :", DATASET_PATH)
print("Exists       :", DATASET_PATH.exists())

## 1 · `load_item_dataset` works + required columns exist


In [ ]:
df = load_item_dataset(DATASET_PATH)

REQUIRED = ["item_text", "transcript_text", "label", "split"]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
print("Required columns present:", REQUIRED)

print(f"\nRows         : {len(df):,}")
print(f"Participants : {df['participant_id'].nunique()}")
print(f"Items        : {sorted(df['item_id'].unique())}")
print(f"Labels       : {sorted(df['label'].unique())}")
print(f"All columns  : {list(df.columns)}")

### Split and label distribution


In [ ]:
print("Split distribution:")
print(df["split"].value_counts().reindex(["train", "validation", "test"]))
print("\nLabel distribution (0-3):")
print(df["label"].value_counts().sort_index())
print("\nLabel distribution per split:")
print(pd.crosstab(df["split"], df["label"]).reindex(["train", "validation", "test"]))

## 2 · Pull one row and inspect


In [ ]:
row = df.iloc[0]
print(f"participant_id : {row['participant_id']}")
print(f"item_id        : {row['item_id']}  ({row['item_name']})")
print(f"label          : {row['label']}")
print(f"split          : {row['split']}")
print()
print("item_text:")
print(f"  {row['item_text']}")
print()
print("transcript_text (first 200 chars):")
print(f"  {' '.join(str(row['transcript_text']).split())[:200]}...")
print()
print("retrieved_utterances (split into list):")
for u in split_utterances(row['retrieved_utterances'])[:5]:
    print(f"  - {u}")

## 3 · Load the tokenizer

MentalBERT, falling back to `bert-base-uncased` if the gated model is unavailable.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "mental/mental-bert-base-uncased"
FALLBACK_MODEL = "bert-base-uncased"
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"Loaded tokenizer: {MODEL_NAME}")
except Exception as e:
    print(f"Could not load {MODEL_NAME}: {e}\nFalling back to {FALLBACK_MODEL}")
    MODEL_NAME = FALLBACK_MODEL
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"Loaded tokenizer: {MODEL_NAME}")

## 4 · `format_model_input` → tokenizer (sentence pair)

`format_model_input` (B3) standardizes any evidence into `[CLS] item_text [SEP] evidence [SEP]`.
The same call works for the full transcript or the retrieved utterances — only the column changes.


In [ ]:
MAX_LEN = 256

for evidence_col in ["transcript_text", "retrieved_utterances"]:
    enc = format_model_input(
        row["item_text"], row[evidence_col],
        tokenizer=tokenizer, max_length=MAX_LEN, return_tensors="pt",
    )
    n_real = int(enc["attention_mask"].sum())
    item_len = int((enc["token_type_ids"][0] == 0).sum())
    print(f"evidence = {evidence_col}")
    print(f"  input_ids shape {tuple(enc['input_ids'].shape)} | "
          f"non-pad {n_real}/{MAX_LEN} | item segment {item_len}")
    print(f"  decoded: {tokenizer.decode(enc['input_ids'][0][:n_real])[:160]}...")
    print()

## 5 · Reusable tokenize function for training

`.map`-able over a Hugging Face `Dataset`; `evidence_column` switches baseline vs. retrieval.


In [ ]:
def tokenize_pair(examples, evidence_column="transcript_text"):
    return tokenizer(
        examples["item_text"],
        examples[evidence_column],
        padding="max_length",
        truncation="only_second",
        max_length=MAX_LEN,
    )

batch = df.head(4)[["item_text", "transcript_text"]].to_dict(orient="list")
out = tokenize_pair(batch)
print("Batch:", len(out["input_ids"]), "examples x", len(out["input_ids"][0]), "tokens")
print("\nB4 validation complete: dataset loads, columns present, rows inspectable, tokenizer-ready.")